# Stag Hunt experiments on Google Colab


## 1. Setup: Drive, paths, helpers

In [ ]:
import os, subprocess, time, signal
from google.colab import drive
drive.mount('/content/drive')

DRIVE = '/content/drive/MyDrive/staghunt'   # zip in, results out (survives disconnects)
CODE  = '/content/stag-hunt-with-RL'        # code on the fast local disk (wiped on disconnect)
os.makedirs(DRIVE, exist_ok=True)

def sh(cmd):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True, executable='/bin/bash')
    out = '\n'.join(l for l in (r.stdout + r.stderr).splitlines()
                    if not any(k in l.lower() for k in ('pygame', 'hello from', 'pkg_resources')))
    print(out)

def sweep_procs():
    # (pid, kind) for our own processes only: 'train' = python train.py ..., 'launcher' = bash run_all.sh
    out = []
    for line in subprocess.run(['ps', '-eo', 'pid,args'], capture_output=True, text=True).stdout.splitlines()[1:]:
        pid, *tok = line.split()
        if tok and 'python' in tok[0] and any(t.endswith('train.py') for t in tok[1:4]):
            out.append((int(pid), 'train'))
        elif len(tok) >= 2 and tok[0].endswith('bash') and tok[1].endswith('run_all.sh'):
            out.append((int(pid), 'launcher'))
        elif tok and tok[0].endswith('xargs') and 'run_one' in line:
            out.append((int(pid), 'xargs'))
    return out

def n_training():
    p = sweep_procs()
    return sum(k == 'train' for _, k in p), any(k == 'launcher' for _, k in p)

sh('echo cores: $(nproc); free -h | head -2')
print('Drive folder:', os.listdir(DRIVE))

## 2. Code and environment


In [ ]:
if not os.path.exists(f'{CODE}/train.py'):
    sh(f'''
    mkdir -p {CODE} && unzip -q -o {DRIVE}/stag-code.zip -d {CODE}
    if [ ! -f {CODE}/train.py ]; then sub=$(dirname $(find {CODE} -name train.py | head -1)); mv "$sub"/* {CODE}/; fi
    ''')
else:
    print('code already unpacked')

if not os.path.exists('/content/Gymnasium-Stag-Hunt'):
    sh('cd /content && git clone -q https://github.com/giorgiofranceschelli/Gymnasium-Stag-Hunt.git')
sh('pip install -q -e /content/Gymnasium-Stag-Hunt')

# outputs live on Drive; the code folder links to them
for d in ['runs', 'logs', 'done', 'results']:
    os.makedirs(f'{DRIVE}/{d}', exist_ok=True)
    p = f'{CODE}/{d}'
    if not os.path.islink(p):
        sh(f'rm -rf {p} && ln -s {DRIVE}/{d} {p}')

sh(f'cd {CODE} && python -c "import torch, gymnasium_stag_hunt, train; print(\'OK\', torch.__version__)"')

plain-DQN runs at maul −5

In [ ]:
cmds = open(CMDS).read().splitlines() if 'CMDS' in dir() else open(f'{DRIVE}/commands.txt').read().splitlines()
extra = []
for m in ['', '--mode separate']:
    out = subprocess.run(f'cd {CODE} && python sweep.py --alphas 0 0.5 --mauls -5 --print-commands -- {m}',
                         shell=True, capture_output=True, text=True).stdout
    extra += [l for l in out.splitlines() if 'train.py' in l and l not in cmds]
open(f'{DRIVE}/commands.txt', 'a').write('\n'.join(extra) + '\n')
print(len(extra), 'added'); print('\n'.join(extra))

## 3. Smoke test (optional, ~1 min)
Expect two `step ...` lines and a `final greedy evaluation` block.

In [ ]:
sh(f'cd {CODE} && python train.py --total-steps 2000 --eval-every 1000 --eval-episodes 5 '
   f'--final-eval-episodes 5 --out /content/test_runs; rm -rf /content/test_runs')

## 4. Run list
Built once and stored on Drive as `commands.txt`, so it cannot change while a sweep is running.
Default-reward sweep: 4 learning rules (hysteretic β = 0.2, NoisyNet per step, NoisyNet per episode, DQN with ε-start 0.1 as control) × α ∈ {0, 0.5} × 3 seeds × {shared, separate} networks = 48 runs.
For separate networks, `--mode separate` is appended after the `--mode shared` that `sweep.py` writes; argparse keeps the last value.

To start a different sweep later, first move the old `commands.txt`, `runs/`, `logs/` and `done/` out of `MyDrive/staghunt/`.

In [ ]:
INCLUDE_SEPARATE = True   # False -> only the 24 shared-network runs

CMDS = f'{DRIVE}/commands.txt'
if os.path.exists(CMDS):
    print('commands.txt already exists, keeping it')
else:
    variants = ['--hyst-beta 0.2', '--noisy', '--noisy --noise-resample episode', '--eps-start 0.1']
    modes = [''] + (['--mode separate'] if INCLUDE_SEPARATE else [])
    lines = []
    for m in modes:
        for v in variants:
            out = subprocess.run(f'cd {CODE} && python sweep.py --alphas 0 0.5 --mauls -5 --print-commands -- {m} {v}',
                                 shell=True, capture_output=True, text=True).stdout
            lines += [l for l in out.splitlines() if 'train.py' in l]
    open(CMDS, 'w').write('\n'.join(lines) + '\n')

cmds = open(CMDS).read().splitlines()
print(len(cmds), 'runs |', sum('separate' in c for c in cmds), 'with separate networks')
print(cmds[0]); print(cmds[-1])

## 5. Launch / resume


In [ ]:
open('/content/run_all.sh', 'w').write(f'''#!/bin/bash
cd {CODE}
export OMP_NUM_THREADS=1 MKL_NUM_THREADS=1
run_one() {{
  i=$1
  [ -f done/$i ] && exit 0
  cmd=$(sed -n "${{i}}p" {DRIVE}/commands.txt)
  echo "$cmd" > logs/run_$i.log
  bash -c "$cmd" >> logs/run_$i.log 2>&1 && touch done/$i
}}
export -f run_one
seq 1 $(wc -l < {DRIVE}/commands.txt) | xargs -P {os.cpu_count()} -I{{}} bash -c 'run_one {{}}'
echo ALL FINISHED
''')

n, launcher = n_training()
if launcher or n > 0:
    print(f'Sweep already running ({n} training processes) - not launching again.')
else:
    log = open(f'{DRIVE}/sweep.log', 'a')
    subprocess.Popen(['bash', '/content/run_all.sh'], stdout=log, stderr=subprocess.STDOUT, start_new_session=True)
    time.sleep(15)
    print('launched - training processes now:', n_training()[0])

## 6. Progress

In [ ]:
n, launcher = n_training()
total = len(open(f'{DRIVE}/commands.txt').read().splitlines())
print(f'finished: {len(os.listdir(f"{DRIVE}/done"))} / {total} | running: {n} | launcher alive: {launcher}')
sh(f'''cd {DRIVE}/logs; for f in $(ls -t *.log 2>/dev/null | head -4); do
      echo "== $f: $(grep '^step' $f | tail -1 | cut -c1-120)"; done
      tail -1 {DRIVE}/sweep.log 2>/dev/null''')

## 7. Aggregate results
When finished = total. Writes `results/summary.csv`.

In [ ]:
sh(f'cd {CODE} && python aggregate.py')

## 8. Behaviour analysis
How the agents cooperate: co-location, rendezvous time, waiting, and whether catches happen as a pair or by converging. Writes `results/behaviour_runs.csv` and `results/behaviour_summary.csv`. No training; a CPU runtime is enough.

In [ ]:
sh(f'cd {CODE} && python analyze_behaviour.py --episodes 200')

ASCII replay of one episode of one run:

In [ ]:
RUN = 'runs/shared_hyst0.2_a0.5-0.5_maul-5_s10002'
sh(f'cd {CODE} && python analyze_behaviour.py --show {RUN} --show-steps 20')

## 9. Export
Packs runs, results and the run list into one zip on Drive (`MyDrive/staghunt/staghunt_results.zip`), to download in one go.

In [ ]:
sh(f'cd {DRIVE} && rm -f staghunt_results.zip && zip -qr staghunt_results.zip runs results commands.txt && ls -lh staghunt_results.zip')

## Emergency only: stop everything
Kills all training. Finished runs stay; unfinished ones restart from zero the next time you run cell 5.

In [ ]:
STOP = False   # set to True and run to stop the sweep
if STOP:
    for pid, kind in sweep_procs():
        try: os.kill(pid, signal.SIGKILL)
        except ProcessLookupError: pass
    time.sleep(3)
    print('training processes left:', n_training()[0])